# Comparison: Head-Only vs Head + LoRA (Frozen Backbone)

### Experimental Setup

We trained a Wav2Vec 2.0-based ASR model under two configurations (both for 75 epochs and the same training args), while keeping the encoder (backbone) frozen:

1. **Head-Only Training**

   * The pretrained encoder is fully frozen.
   * Only the classification head is trained.

2. **Head + LoRA Training**

   * The encoder remains frozen.
   * Low-Rank Adaptation (LoRA) modules are inserted into the encoder.
   * Both the classification head and LoRA parameters are trained.

---

### Motivation

Freezing the backbone reduces computational cost and preserves pretrained knowledge. However, it limits adaptability to new domains. LoRA addresses this by introducing a small number of trainable parameters that can **adapt internal representations without full fine-tuning**.

---

### Quantitative Results

| Model Configuration | Validation Loss | WER ↓    | CER ↓     |
| ------------------- | --------------- | -------- | --------- |
| Head Only           | 3.87            | 0.67     | 0.38      |
| Head + LoRA         | **1.577**       | **0.35** | **0.128** |

**Key Observations:**

* LoRA reduces validation loss by more than **2×**
* WER improves by ~**48% relative**
* CER improves by ~**66% relative**

---

### Qualitative Comparison

#### Reference

```
allow me to explain systemd an essential init system for linux operating systems it is responsible for starting and managing services during boot up by utilizing units defined in configuration files systemd organizes service dependencies effectively furthermore this framework includes features such as logging and service management under its own scope of control making systemd integral to modern linux distributions today
```

---

#### Head-Only Output

```
allow me to explain system d an essential init system for linox operating systems it is responsible for starting and managing services during butto by utilizing units to fint in configuration files system d organizes service de pendencies effectively furthermore this framer concludes features such as logging and service management under its own scope of control making system d integral to modern linox distributions to day
```

**Error Patterns:**

* Frequent **phonetic errors**: *linux → linox*, *boot → butto*
* **Token fragmentation**: *systemd → system d*, *dependencies → de pendencies*
* **Semantic drift**: *framework includes → framer concludes*
* Poor handling of **technical vocabulary**

---

#### Head + LoRA Output

```
allow me to explain systemd an essential in its system for linux operating systems it is responsible for starting and managing services during bootup by utilizing units to find inconfiguration files system d organizes service dependencies effectively furthermore this framework concludes features such as logging and service management under its own scope of control making system d integral to modern linux distributions today
```

**Improvements:**

* Correct recognition of **domain-specific terms** (*systemd*, *linux*)
* Better **word cohesion** (*boot up → bootup*)
* Reduced **phonetic confusion**
* More stable **sentence structure**

**Remaining Issues:**

* Minor grammatical slips (*init → in its*)
* Occasional spacing errors (*inconfiguration*)
* Some residual inconsistencies (*system d*)

---

### Analysis

The head-only approach relies entirely on **fixed acoustic representations**, limiting its ability to adapt to:

* Domain-specific vocabulary (e.g., "systemd")
* Subtle phonetic distinctions
* Token boundary consistency

In contrast, LoRA introduces **trainable low-rank updates inside the encoder**, enabling:

* Adaptation of intermediate representations
* Improved alignment between audio and token space
* Better generalization to technical speech

Despite adding minimal computational overhead (~6 minutes training difference), LoRA yields **substantial gains across all metrics**.

---

### Conclusion

Adding LoRA to a frozen Wav2Vec 2.0 encoder significantly improves ASR performance compared to training the head alone.

* **Much lower loss**
* **Dramatically improved WER/CER**
* **Better handling of technical language**
* **Negligible training overhead**

This demonstrates that **parameter-efficient fine-tuning (PEFT)** methods like LoRA are highly effective for adapting pretrained speech models under resource constraints.